In [1]:
#Imports and let notebook find code
import sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

import pandas as pd
from config import load_config
cfg = load_config()
print("Settings:", cfg["embedding_model"], "| top_k =", cfg["retrieval"]["top_k"])

Settings: sentence-transformers/all-MiniLM-L6-v2 | top_k = 3


In [2]:
#Split sections into chunks
from chunking import build_chunks
chunks = build_chunks(cfg)
print(len(chunks), "chunks | longest:", chunks.char_count.max(), "characters")
display(chunks.groupby("jurisdiction").size())
display(chunks[chunks.label.str.contains("vacation entitlement year")][["chunk_id", "label", "text"]])

1614 chunks | longest: 1068 characters


jurisdiction
British Columbia     471
Ontario             1143
dtype: int64

,chunk_id,label,text
2,ON-ESA-1-3,1(1) “alternative vacation entitlement year”,"“alternative vacation entitlement year” means,..."
43,ON-ESA-1-44,1(1) “standard vacation entitlement year”,"“standard vacation entitlement year” means, wi..."
51,ON-ESA-1-52,1(1) “vacation entitlement year”,“vacation entitlement year” means an alternati...


In [3]:
#Chunks into numbers and stores them into FAISS
from build_index import build_index
build_index(cfg)

W1103 17:43:48.364000 19198 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

{'embedding_model': 'sentence-transformers/all-MiniLM-L6-v2',
 'dimensions': 384,
 'chunks': 1614,
 'by_jurisdiction': {'Ontario': 1143, 'British Columbia': 471},
 'build_seconds': 7.1,
 'device': 'mps:0',
 'index_mb': 2.48}

In [4]:
#Testing searches
from retrieval import Retriever
r = Retriever(cfg)
for q, jur in [("How much vacation pay do I get after 5 years?", "British Columbia"),
               ("When does overtime start?", "Ontario"),
               ("I've been working in Toronto for about 3 years. How much paid time off am I supposed to get each year?", "Ontario")]:
    print("\n", jur, "|", q)
    display(r.search(q, jur, k=3)[["rank", "score", "label", "text"]])

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]


 British Columbia | How much vacation pay do I get after 5 years?


,rank,score,label,text
0,1,0.5807,58,(1) An employer must pay an employee the follo...
1,2,0.4962,52.5(5.1),"(5.1) Subject to subsection (5.2), an employer..."
2,3,0.4705,49.1(3),"(3) Subject to subsection (4), an employer mus..."



 Ontario | When does overtime start?


,rank,score,label,text
0,1,0.4836,22(8),(8) If the employment of an employee ends befo...
1,2,0.4608,22(3),"(3) Subject to subsections (3.1) and (3.2), an..."
2,3,0.4577,15(1),(1) An employer shall record the following inf...



 Ontario | I've been working in Toronto for about 3 years. How much paid time off am I supposed to get each year?


,rank,score,label,text
0,1,0.5703,15(4),(4) An employee is considered to be paid a sal...
1,2,0.5518,50.1(1.3),"(1.3) Subject to subsection (1.4), an employee..."
2,3,0.5425,63(2.4),(2.4) For the purposes of subsection (2.3) and...


In [5]:
#Show overall scores
from eval_retrieval import evaluate
summary, details = evaluate(r, cfg)
print("Overall:", summary["overall"])
display(pd.DataFrame(summary["by_phrasing"]).T)
display(pd.DataFrame(summary["by_category"]).T)
display(pd.DataFrame(summary["misses_at_top_k"]))

Overall: {'recall@1': 0.553, 'recall@3': 0.776, 'recall@5': 0.868}


,recall@1,recall@3,recall@5
casual,0.560,0.680,0.820
precise,0.538,0.962,0.962


,recall@1,recall@3,recall@5
cross_jurisdiction_comparison,0.562,0.938,1.000
dates_thresholds,0.667,0.667,0.833
direct_extraction,0.600,0.600,0.800
exceptions,0.800,1.000,1.000
insufficient_evidence,0.000,0.375,0.375


,id,jurisdiction,question,gold,retrieved
0,Q01,Ontario,I've been working in Toronto for about 3 years...,[33],"[15(4) (0.57), 50.1(1.3) (0.55), 63(2.4) (0.54..."
1,Q05,Ontario,I work in Toronto and sometimes end up doing 4...,[22],"[11(5) (0.54), 18 (0.53), 27(2) (0.53), 30(2) ..."
2,Q06,British Columbia,I just finished my first year working in BC. A...,[57],"[1(1) *pay period* (0.52), 52.1(4) (0.52), 49...."
3,Q08,British Columbia,I'm working a 10-hour shift in BC today. Shoul...,[40],"[37(5) (0.54), 37(4) (0.54), 34(3) (0.53), 37(..."
4,Q17,Ontario,I want to split my vacation into single days i...,[35],"[35.1(2) (0.63), 33(3) (0.61), 34(5) (0.61), 3..."
5,Q24,Ontario,"I started my job in Toronto on March 15, 2025....","[1, 35]","[33(1) (0.69), 33(4) (0.69), 34(5) (0.69), 35 ..."
6,Q28,British Columbia,Are employees in managerial positions exempt f...,"[37, 40]","[39 (0.59), 37(7) (0.58), 37(3) (0.57), 37(10)..."
7,Q29,Ontario,My sister works in Calgary and I work in Toron...,[35.2],"[39 (0.59), 36(2) (0.59), 36(4) (0.58), 36(1) ..."
8,Q30,British Columbia,I'm a software developer in Vancouver. Do tech...,[40],"[35 (0.53), 1(1) *overtime wages* (0.52), 39 (..."


In [6]:
#Abstain threshold
import sys
!{sys.executable} ../src/abstain_threshold.py

W1103 21:13:42.282000 20695 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.
Loading weights: 100%|████████████████████| 103/103 [00:00<00:00, 13996.87it/s]
{
  "floor": 0.494,
  "floor_rule": "5th percentile of top-1 similarity on 92 answerable train/val questions",
  "nontest_answerable": {
    "n": 92,
    "min": 0.378,
    "median": 0.685,
    "max": 0.873
  },
  "test_answerable": {
    "n": 35,
    "min": 0.474,
    "median": 0.655,
    "max": 0.859
  },
  "test_should_abstain": {
    "n": 3,
    "min": 0.532,
    "median": 0.588,
    "max": 0.594
  },
  "abstain_caught": "0/3",
  "answerable_wrongly_flagged": "1/35"
}
abstain pairs: [('Q27', 'Ontario', 0.588), ('Q28', 'British Columbia', 0.594), ('Q30', 'British Columbia', 0.532)]
wrongly flagged: [('Q11', 'British Columbia', 0.474)]
Wrote /Users/harris/Documents/government-policy-assistant/results/abstain_threshold.json


In [7]:
#Leakage Check:
!{sys.executable} ../src/check_leakage.py

W1103 21:15:14.567000 20705 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.
Loading weights: 100%|████████████████████| 103/103 [00:00<00:00, 12838.05it/s]
PASS  1_all_excerpts_from_train_pool
PASS  2_no_test_gold_sections_in_training
PASS  3_train_val_section_overlap
PASS  4_no_13gram_overlap_with_held_out_law
PASS  5_no_near_duplicate_questions
closest question pairs:
  0.716  T061 ↔ Q25: 'I started a new job in BC three weeks ago. Do I get paid for the upcoming statutory holiday?' / "I started a new job in BC last week. How long do I have to work before I'm entitled to vacation pay?"
  0.690  T006 ↔ Q19: 'Does my employer in Ontario have to pay me on a regular schedule?' / 'Instead of getting my vacation pay all at once, can my employer in Ontario just add it to every paycheque?'
  0.689  T008 ↔ Q23: 'Is my employer in Ontario allowed to take money off my paycheque?' / 'I work in Ontario and would rather take time off 

In [8]:
#Base QWEN smoke test:
!{sys.executable} ../src/smoke_test.py

W1103 21:16:02.394000 20710 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.
Loading weights: 100%|██████████████████████| 338/338 [00:02<00:00, 141.64it/s]

=== V003 (126 tokens, 8.1s) ===
{
  "jurisdiction": "Ontario",
  "topic": "Employment standards",
  "section": "s. 49.6(2)",
  "eligibility": "An employee who has been employed by an employer for at least six consecutive months",
  "entitlement_or_requirement": "leave of absence without pay of up to 104 weeks",
  "threshold": "if a child of the employee disappears and it is probable, considering the circumstances, that the child disappeared as a result of a crime",
  "exceptions": [],
  "mandatory": "mandatory"
}
checks: {'valid_json': True, 'exact_keys': False, 'section_match': False, 'mandatory_match': True, 'quote_word_for_word': False, 'says_not_specified_when_expected': True}

=== V008 (106 tokens, 6.5s) ===
{
  "jurisdiction": "Ontario",
  "topic": "Employment s